# Train DustyLM from Scratch

Train Dusty, a tiny ~8M parameter assistant that talks like a robot vacuum. From raw data to a chatting model.

**What you will do:**
1. Set up the environment (Colab, RunPod, or local)
2. Download the training datasets
3. Train the tokenizer
4. Prepare tokenized data
5. Pretrain the base model
6. Select the best base checkpoint
7. Fine-tune with SFT
8. Select the best SFT checkpoint
9. Chat with your trained model

This notebook works on **Google Colab, RunPod, or any local machine**. The setup cell auto-detects your environment.

> **Note on Execution:** This notebook is designed as an interactive, educational guide.
> It includes manual checkpoint evaluations and is not meant to be run top-to-bottom in a single click.
> If you are looking for a fully automated end-to-end run, open your terminal and execute `make train-end-to-end`.


In [ ]:
import sys, os
from pathlib import Path
import torch

if "google.colab" in sys.modules:
    !git clone --depth 1 https://github.com/khordoo/dusty-lm.git
    os.chdir("dusty-lm")
    !pip install -q -e .
elif not Path("Makefile").exists():
    # Walk up so make commands find the Makefile from any CWD
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "Makefile").exists():
            os.chdir(str(parent))
            break
    else:
        print("Run this from the repo root:\n        uv sync")
REPO_ROOT = Path.cwd()
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print("Repo root:", REPO_ROOT)
print("PyTorch:", torch.__version__)
print("Training device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Download the Datasets

Downloads TinyStories for pretraining and the Dusty SFT JSONL from [mkhordoo/dusty-chat](https://huggingface.co/datasets/mkhordoo/dusty-chat). If you already have `artifacts/datasets/tinystories_base.txt` and `artifacts/datasets/dusty_sft.jsonl`, skip this cell.

**Why TinyStories?**
The sole purpose of pre-training is to teach basic English grammar and logic. Because our custom tokenizer is intentionally constrained to 4,000 tokens, we need simplified language. The Dusty vacuum persona naturally aligns with the direct, limited vocabulary of a 3-to-4-year-old. TinyStories fits these constraints perfectly, letting the 8M model learn correct English structure without being overwhelmed by complex syntax.

By default this downloads a 100k subset of TinyStories for a fast smoke test. To download the full dataset, pass `TINYSTORIES_SLICE="train"`:
`!make download-datasets TINYSTORIES_SLICE=train`

In [ ]:
!make download-datasets

## 2. Check the Training Files

Training needs TinyStories text and SFT data under `artifacts/datasets/`. This cell verifies the required files exist.

In [ ]:
required_sources = {
    "pretrain text": REPO_ROOT / "artifacts" / "datasets" / "tinystories_base.txt",
    "SFT chat JSONL": REPO_ROOT / "artifacts" / "datasets" / "dusty_sft.jsonl",
}

for label, path in required_sources.items():
    status = "✓ found" if path.exists() else "✗ missing"
    print(f"{label:16} {status:10} {path}")

## 3. Train the Tokenizer

Dusty uses a small 4096-token BPE vocabulary. This trains the tokenizer from the local Dusty text and writes `artifacts/tokenizers/dusty_tokenizer.json'.

> **🚨 Important:** Tokenizer training happens **only** at the very beginning of the pre-training phase. You must **never** retrain the tokenizer during or before the SFT phase. The base model's weights are permanently mapped to specific token IDs; retraining shuffles these IDs, causing the model to output pure garbage. Treat the tokenizer as a strictly fixed artifact for the entire remaining pipeline.


In [ ]:
!make tokenizer

## 4. Prepare Tokenized Datasets

Turns raw text and chat JSONL into Hugging Face `datasets` saved on disk. The training loop reads these tokenized datasets directly.

In [ ]:
!make data-pretrain
!make data-sft

## 5. Pretrain Dusty

Pretraining teaches the base `dusty8m` model language patterns before SFT teaches it to answer user messages. `EPOCHS=1` is a smoke test; increase for a better model.

> **Note:** We stopped pretraining at 5,700 steps (~46M tokens, ~2 epochs) because the model was already generating cohesive English. While Chinchilla scaling laws suggest ~19,600 steps (160M tokens) for an 8M parameter model, this early checkpoint works well for this lightweight demonstration.

**Start TensorBoard first** (non-blocking, runs in background), then start training.


## 5b. Monitor Training with TensorBoard

The `%tensorboard` magic is non-blocking: it starts the dashboard as a background process and renders the UI immediately. You can run the training cell right after without waiting.

TensorBoard auto-refreshes every 30 seconds as new metrics arrive.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir artifacts/tensorboard


In [ ]:
!make train-pretrain EPOCHS=1 CHECKPOINT_EVERY_STEPS=50

### 5c. The Full Training Lifecycle Loss Curve

Below is the TensorBoard loss curve from our production runs, showing both phases of training overlaid:

* **Pink Line (Pre-training):** We intentionally implemented early stopping around step 5,700. Monitoring showed the model was already outputting cohesive English, meaning further compute burn wasn't necessary for this baseline.
* **Yellow Line (SFT):** The fine-tuning phase ran for a full ~19,600 steps to strictly enforce the robot vacuum persona and conversational formatting.

If you are running this locally, spin up TensorBoard to ensure your pre-training curve follows a similar downward trajectory before moving to SFT.

<img src="../docs/images/training-lifecycle-loss.png" alt="Dusty pretraining loss curve" width="600"/>


## 6. Check the Pretrained Base Model

Before SFT, generate from the `dusty8m` base profile. This tells you whether the base checkpoint has learned stable text patterns.

In [ ]:
!make generate PROFILE=dusty8m PROMPT="deep in the forest, there lived a " TOP_P=0.9 TEMPERATURE=0.8

## 7. Compare Pretraining Step Checkpoints

If you saved step checkpoints, generate from a few candidates. Pick the checkpoint with the best stability and Dusty-like rhythm before SFT. Replace the step numbers with checkpoints that exist in your `artifacts/checkpoints/` folder.

In [ ]:
for step in [50, 100, 150]:
    print("=" * 80)
    print("CHECKPOINT_STEP:", step)
    !make generate PROFILE=dusty8m CHECKPOINT_STEP={step} PROMPT="deep in the forest, there lived a " TOP_P=0.9 TEMPERATURE=0.8

## 8. Promote the Best Base Checkpoint

SFT initializes from `artifacts/checkpoints/dusty8m.pt`. If a step checkpoint is better than the final one, copy it before running SFT.

In [ ]:
BEST_PRETRAIN_STEP = 100

!cp artifacts/checkpoints/dusty8m_step_{BEST_PRETRAIN_STEP}.pt artifacts/checkpoints/dusty8m.pt

# Verify the promoted checkpoint
!make generate PROFILE=dusty8m PROMPT="deep in the forest, there lived a " TOP_P=0.9 TEMPERATURE=0.8

## 9. Fine-Tune with SFT

The `sft_dusty8m` profile automatically initializes from `artifacts/checkpoints/dusty8m.pt`. `EPOCHS=1` is a smoke test. For our configuration, the sweet spot was around step 19,600 (~21 epochs at 938 steps per epoch).

In [ ]:
!make train-sft EPOCHS=1 CHECKPOINT_EVERY_STEPS=50

### Training Reference

Here are the numbers from our production runs:

| Metric | Pretrain | SFT |
|---|---|---|
| Dataset examples | 87,586 | 30,000 |
| Batch size | 32 | 32 |
| Max seq len | 256 | 256 |
| Steps per epoch | ~2,738 | ~938 |
| Total steps run | ~5,700 | ~21,500 |
| Best checkpoint | ~5,700 | ~19,600 |
| Tokens seen | ~46M | ~160M |
| Epochs | ~2.1 | ~23 |

If you train with these settings, your results should be similar.

## 10. Compare SFT Step Checkpoints

With `EPOCHS=1` the SFT run produces ~938 steps. Step checkpoints at 200, 700, and 900 are saved in `artifacts/checkpoints/`. Generate from them below to see how the model evolves.

> **Note:** These are very early checkpoints. The model learns to respond to questions at this stage but has not yet learned to answer correctly. A production run needs ~19,000 steps (~21 epochs) to converge on a consistent Dusty persona. The steps below show the model's progression from barely structured responses toward recognizing the chat format — still far from polished.

### Understanding Generation Parameters

An 8M model has a noisy vocabulary tail. Restricting its choices is critical for coherent output. These settings apply to every `make generate` command in this notebook:

| Parameter | Value | Effect |
|---|---|---|
| Temperature | 0.6 - 0.8 | Controls randomness. Lower (0.1 - 0.3) is repetitive; above 1.0 produces gibberish. |
| Top-K | 5 | Forces the model to only consider its 5 most likely tokens, cutting off nonsense. |
| Top-P | 0.8 - 0.9 | Nucleus sampling. Only tokens making up the top 80-90% of probability mass are considered. |

These defaults are set in the `sft_dusty8m` profile inside `dustylm/config.py`. You can override them per command (as done here with `TOP_P` and `TEMPERATURE`) or edit the profile permanently.

In [ ]:
for step in [200, 700, 900]:
    print("=" * 80)
    print("SFT CHECKPOINT_STEP:", step)
    !make generate PROFILE=sft_dusty8m CHECKPOINT_STEP={step} PROMPT="who are you?" TOP_P=0.9 TEMPERATURE=0.8

## 11. Promote the Best SFT Checkpoint

The chat cell below loads `artifacts/checkpoints/dusty8m_sft.pt`. Replace it with your best step checkpoint by copying the step file to that path, then chat with the promoted model.

In [ ]:
BEST_SFT_STEP = 19600

!cp artifacts/checkpoints/dusty8m_sft_step_{BEST_SFT_STEP}.pt artifacts/checkpoints/dusty8m_sft.pt

# Verify the promoted checkpoint has the right personality and coherent responses
!make generate PROFILE=sft_dusty8m PROMPT="who are you?" TOP_P=0.9 TEMPERATURE=0.8

## 12. Chat with Your Trained Model

In [ ]:
from dustylm.inference import Inference

engine = Inference(
    checkpoint_path=REPO_ROOT / "artifacts" / "checkpoints" / "dusty8m_sft.pt",
    tokenizer_path=REPO_ROOT / "artifacts" / "tokenizers" / "dusty_tokenizer.json",
    device=device,
)


def chat(prompt):
    response = engine.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=64,
        temperature=0.6,
        top_p=0.8,
    )
    return response["choices"][0]["message"]["content"].strip()


for prompt in ["who are you?", "what makes you happy"]:
    print(f"You> {prompt}")
    print(f"Dusty> {chat(prompt)}\n")

## Save Your Checkpoints

Hosted runtimes (Colab, RunPod) can disappear. Download checkpoints or copy them to persistent storage before shutting down.

In [ ]:
from pathlib import Path

for path in sorted(Path("artifacts/checkpoints").glob("*.pt")):
    print(path, f"{path.stat().st_size / 1024 / 1024:.1f} MB")

---

## Advanced: Generate Custom Training Data

The repo can generate synthetic data via OpenRouter. These steps may call an external model provider, so set your API key first. Once data exists, training is just the `make` commands above. This topic is covered in more detail in `notebooks/03_advanced_tools.ipynb`.

In [ ]:
# Optional: generate SFT chat examples.
# Uncomment this when your model/API credentials are configured.
# !make synthesize-sft

### Changing Dusty's Persona

If you simply want to retrain Dusty, use the downloaded dataset from `make download-datasets` and keep the training flow unchanged.

If you want a different robot persona, create a new SFT dataset by editing the prompt inside `data_pipeline/generate_sft.py`. Change the instructions that define Dusty's personality, topics, and speaking style, then run `make synthesize-sft` to produce your new chat JSONL before training.

## Quick Reference

Run these in a terminal when you do not need the notebook:

```bash
make download-datasets
make tokenizer
make data-pretrain
make train-pretrain EPOCHS=1
make data-sft
make train-sft EPOCHS=1
make chat
```

That is the whole loop: download data → tokenizer → pretrain → SFT → chat.

### Run End-to-End

```bash
make train-end-to-end
```

This runs the full pipeline with `EPOCHS=1` hardcoded, ideal as a smoke test to verify all stages run correctly. For real training with more epochs, use the individual commands above so you can monitor the loss curve at each stage.